# Projeto Churn SaaS - EDA + Modelagem + Insights

Este notebook concentra toda a entrega do desafio da TechGrow em um unico lugar.

Contexto:
- objetivo de negocio: reduzir cancelamento (churn)
- objetivo analitico: entender drivers de churn e prever clientes com maior risco
- target: `churn` (`1 = cancelou`, `0 = manteve`)

## Roteiro da analise

1. Carregar e explorar os dados
2. Verificar qualidade (tipos, nulos, estatisticas)
3. EDA visual para entender padroes de churn
4. Preparar dados (encoding, scaling, remocao de redundancia)
5. Split treino/teste (70/30)
6. Treinar 3 modelos: Logistic Regression, Random Forest e XGBoost
7. Avaliar com accuracy, precision, recall, f1-score e roc_auc_score
8. Interpretar variaveis mais importantes e transformar em acoes de negocio

In [ ]:
# As métricas dependem da partição de treino/teste e das versões do ambiente; preservar essas condições permite comparações.
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
TARGET = "churn"
ID_COL = "id_cliente"

## 1) Carga inicial dos dados

Aqui validamos rapidamente estrutura da base, tipos e amostra de registros.

In [ ]:
df = pd.read_csv("data/Caso_Pratico_base_churn_saas.csv")

print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")

display(df.head())
display(df.dtypes.to_frame("dtype"))

## 2) Qualidade dos dados

Nesta etapa verificamos:
- valores ausentes
- estatisticas descritivas
- taxa geral de churn

In [ ]:
missing_df = df.isna().sum().rename("missing_values").to_frame()
display(missing_df)

stats_df = df.describe(include="all").T
display(stats_df)

print(f"Taxa de churn geral: {df[TARGET].mean():.2%}")

## 3) EDA - distribuicao do target

Este grafico mostra equilibrio (ou nao) entre clientes que cancelaram e os que permaneceram.

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x=TARGET, palette=["#4C78A8", "#F58518"])
total = len(df)
for patch in ax.patches:
    value = int(patch.get_height())
    pct = 100 * value / total
    ax.annotate(
        f"{value} ({pct:.1f}%)",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
    )
ax.set_title("Distribuicao de churn")
ax.set_xlabel("churn")
ax.set_ylabel("clientes")
plt.show()

## 4) Separacao de variaveis

Definimos colunas numericas e categoricas para facilitar visualizacoes e pipeline de modelagem.

In [ ]:
feature_cols = [c for c in df.columns if c not in [TARGET, ID_COL]]
num_cols = df[feature_cols].select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

print("Variaveis numericas:")
print(num_cols)
print("\nVariaveis categoricas:")
print(cat_cols)

## 5) EDA - distribuicoes numericas por churn

Comparacao visual entre churn `0` e churn `1` para cada variavel numerica.

In [ ]:
ncols = 2
nrows = int(np.ceil(len(num_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols):
    ax = axes[i]
    ax.hist(df[df[TARGET] == 0][col], bins=16, alpha=0.65, label="churn=0", color="#4C78A8")
    ax.hist(df[df[TARGET] == 1][col], bins=16, alpha=0.65, label="churn=1", color="#F58518")
    ax.set_title(f"Distribuicao: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("frequencia")
    ax.legend()

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 6) EDA - boxplots por churn

Boxplots ajudam a identificar diferencas de mediana, dispersao e outliers entre classes.

In [ ]:
ncols = 2
nrows = int(np.ceil(len(num_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols):
    ax = axes[i]
    sns.boxplot(data=df, x=TARGET, y=col, ax=ax, palette=["#4C78A8", "#F58518"])
    ax.set_title(f"Boxplot por churn: {col}")
    ax.set_xlabel("churn")
    ax.set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 7) EDA - churn por variaveis categoricas

Mostra taxa media de churn por categoria (`plano`, `regiao`, etc.).

In [ ]:
if cat_cols:
    ncols = 2
    nrows = int(np.ceil(len(cat_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(cat_cols):
        churn_rate = df.groupby(col, as_index=False)[TARGET].mean().sort_values(TARGET, ascending=False)
        display(churn_rate)

        ax = axes[i]
        sns.barplot(data=churn_rate, x=col, y=TARGET, ax=ax, color="#4C78A8")
        ax.set_title(f"Taxa de churn por {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("media churn")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=25)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

## 8) EDA - correlacao e colunas redundantes

Para evitar excesso de informacao repetida, removemos colunas numericas muito correlacionadas
(limiar de correlacao absoluta >= 0.92).

In [ ]:
corr = df[num_cols + [TARGET]].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de correlacao (numericas + churn)")
plt.show()

threshold = 0.92
pairs = []
redundant = set()
abs_corr = df[num_cols].corr(numeric_only=True).abs()

for i, c1 in enumerate(num_cols):
    for j in range(i + 1, len(num_cols)):
        c2 = num_cols[j]
        c12 = abs_corr.loc[c1, c2]
        if c12 >= threshold:
            c1_target = abs(df[c1].corr(df[TARGET]))
            c2_target = abs(df[c2].corr(df[TARGET]))
            drop = c1 if c1_target < c2_target else c2
            keep = c2 if drop == c1 else c1
            redundant.add(drop)
            pairs.append(
                {
                    "feature_a": c1,
                    "feature_b": c2,
                    "abs_corr": c12,
                    "keep": keep,
                    "drop": drop,
                }
            )

pairs_df = pd.DataFrame(pairs)
display(pairs_df)
redundant = sorted(redundant)
print("Colunas redundantes removidas:", redundant)

## 9) Preparacao para modelagem

- removemos `id_cliente` (identificador)
- mantemos colunas relevantes apos filtro de redundancia
- split treino/teste: 70/30 com `stratify`

In [ ]:
num_model = [c for c in num_cols if c not in redundant]
model_features = num_model + cat_cols

X = df[model_features].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print(f"Taxa churn treino: {y_train.mean():.2%}")
print(f"Taxa churn teste: {y_test.mean():.2%}")

## 10) Treino dos modelos

Pipeline adotado:
- numericas: imputacao mediana + padronizacao
- categoricas: imputacao mais frequente + One-Hot Encoding

Modelos:
1. Logistic Regression (baseline)
2. Random Forest
3. XGBoost

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_model,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            cat_cols,
        ),
    ]
)

models = {
    "logistic_regression": LogisticRegression(max_iter=1200, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "xgboost": XGBClassifier(
        n_estimators=350,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
}

results = []
preds = {}
fitted = {}

for name, model in models.items():
    pipeline = Pipeline([("preprocess", preprocess), ("model", model)])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1_score": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc_score": roc_auc_score(y_test, y_prob),
        }
    )

    preds[name] = {"y_pred": y_pred, "y_prob": y_prob}
    fitted[name] = pipeline

metrics_df = pd.DataFrame(results).sort_values("roc_auc_score", ascending=False).reset_index(drop=True)
display(metrics_df)

## 11) Avaliacao visual

### 11.1 Matrizes de confusao
Permitem avaliar erros de classificacao (falsos positivos e falsos negativos).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, models.keys()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        preds[name]["y_pred"],
        ax=ax,
        cmap="Blues",
        colorbar=False,
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

### 11.2 Curvas ROC

Comparam a capacidade de separacao entre classes em diferentes thresholds.

In [ ]:
plt.figure(figsize=(8, 6))

for name in models.keys():
    RocCurveDisplay.from_predictions(y_test, preds[name]["y_prob"], name=name)

plt.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
plt.title("Curvas ROC")
plt.show()

## 12) Interpretacao - variaveis que mais impactam churn

Consolidamos importancias dos 3 modelos para obter uma visao mais robusta dos drivers.

In [ ]:
def extract_importance(pipe, model_name, cat_features):
    feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
    model = pipe.named_steps["model"]

    if model_name == "logistic_regression":
        values = model.coef_.ravel()
    else:
        values = model.feature_importances_

    imp = pd.DataFrame(
        {
            "feature": feat_names,
            "importance": values,
            "abs_importance": np.abs(values),
        }
    )

    def base_feature(name):
        if name.startswith("num__"):
            return name.replace("num__", "", 1)
        if name.startswith("cat__"):
            suffix = name.replace("cat__", "", 1)
            for c in cat_features:
                if suffix.startswith(f"{c}_"):
                    return c
            return suffix.split("_", 1)[0]
        return name

    imp["base_feature"] = imp["feature"].map(base_feature)
    imp["normalized_abs"] = imp["abs_importance"] / imp["abs_importance"].sum()
    imp["model"] = model_name
    return imp.sort_values("abs_importance", ascending=False)

importance_all = []
for model_name, pipe in fitted.items():
    importance_all.append(extract_importance(pipe, model_name, cat_cols))

importance_all = pd.concat(importance_all, ignore_index=True)

consensus = (
    importance_all.groupby(["model", "base_feature"], as_index=False)["normalized_abs"].sum()
    .groupby("base_feature", as_index=False)["normalized_abs"]
    .mean()
    .sort_values("normalized_abs", ascending=False)
)

print("Ranking consolidado de impacto no churn:")
display(consensus)

logistic_imp = importance_all[importance_all["model"] == "logistic_regression"].copy()

print("Top efeitos positivos para churn (Regressao Logistica):")
display(logistic_imp.sort_values("importance", ascending=False).head(8)[["feature", "importance"]])

print("Top efeitos negativos para churn (Regressao Logistica):")
display(logistic_imp.sort_values("importance", ascending=True).head(8)[["feature", "importance"]])

## 13) Conclusoes e recomendacoes de negocio

Leitura esperada apos executar o notebook:

1. Quais variaveis mais impactam churn:
- use o `consensus` como ranking principal
- use os coeficientes da regressao para entender direcao (aumenta/reduz risco)

2. O modelo identifica bem clientes que cancelam?
- priorize `recall` para reduzir falsos negativos (clientes de risco nao identificados)
- combine com `precision` para evitar excesso de alarmes falsos
- acompanhe `roc_auc_score` para visao geral de separacao

3. Acoes recomendadas para Produto/CS:
- priorizar clientes com sinais de baixa adocao e alto atrito
- criar fluxo preventivo para risco financeiro (inadimplencia)
- fortalecer onboarding dos primeiros meses
- acionar playbooks de retencao por segmento (plano/regiao)
- operacionalizar score de churn no CRM para acao proativa

Nota importante:
- se a variavel `inadimplencia` separar churn quase perfeitamente, investigue possivel leakage
  (ou regra de negocio muito deterministica) antes de levar o modelo para producao.